# Real Estate Recommender — LightFM + Content Hybrid
Trains a LightFM model with WARP loss on implicit likes, then blends its collaborative scores
with content-based cosine similarity over listing numeric features.

**Requires:** `pip install lightfm`

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

try:
    import lightfm
except ImportError:
    lightfm = None
    print("lightfm not installed. Run: pip install lightfm")

lightfm not installed. Run: pip install lightfm


In [2]:
# Load datasets
users_df = pd.read_csv('data-refined/users.csv')
listings_df = pd.read_csv('data-refined/cleaned_listings.csv')
user_likes_df = pd.read_csv('data-refined/user_likes.csv')

print(f"Users: {users_df.shape[0]} | Listings: {listings_df.shape[0]} | Likes: {user_likes_df.shape[0]}")

Users: 200 | Listings: 26648 | Likes: 26546


## Content feature matrix
Build an L2-normalised numeric feature matrix over listings that appear in user likes.

In [3]:
# Keep only listings present in user_likes
if 'listing_id' not in listings_df.columns:
    listings_df = listings_df.reset_index().rename(columns={'index': 'listing_id'})

liked_item_ids = user_likes_df['listing_id'].unique()
listings_in_likes = listings_df[listings_df['listing_id'].isin(liked_item_ids)].copy()
missing_items = set(liked_item_ids) - set(listings_in_likes['listing_id'])
if missing_items:
    warnings.warn(f'{len(missing_items)} listing_ids in user_likes not found in listings_df; they will be skipped.')

In [4]:
# Build numeric feature matrix for content similarity
numeric_cols = [c for c in ['price', 'bed', 'bath', 'acre_lot', 'house_size'] if c in listings_in_likes.columns]
if not numeric_cols:
    raise ValueError('No numeric listing features available for content similarity.')
features_raw = listings_in_likes[numeric_cols].fillna(0)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_raw)
item_similarity = cosine_similarity(features_scaled)
print(f"Item similarity matrix: {item_similarity.shape}")

Item similarity matrix: (16791, 16791)


## LightFM WARP model

In [5]:
# Fit LightFM dataset with aligned item ids
if lightfm is None:
    print("lightfm not installed — skipping LightFM section. Install via: pip install lightfm")
else:
    from lightfm import LightFM
    from lightfm.data import Dataset as LFDataset
    from lightfm.cross_validation import random_train_test_split

    user_likes_aligned = user_likes_df[user_likes_df['listing_id'].isin(listings_in_likes['listing_id'])]
    lf_dataset = LFDataset()
    lf_dataset.fit(
        users=user_likes_aligned['user_id'].unique(),
        items=listings_in_likes['listing_id'].unique()
    )
    interactions, _ = lf_dataset.build_interactions(
        user_likes_aligned[['user_id', 'listing_id']].itertuples(index=False, name=None)
    )
    train_interactions, test_interactions = random_train_test_split(
        interactions, test_percentage=0.2, random_state=42
    )
    lf_model = LightFM(no_components=32, learning_rate=0.05, loss='warp', random_state=42)
    lf_model.fit(train_interactions, epochs=20, num_threads=4)
    print("LightFM model trained.")

lightfm not installed — skipping LightFM section. Install via: pip install lightfm


## Evaluation — AUC, Precision@K, Recall@K

In [6]:
# Evaluate LightFM implicit ranking metrics on held-out set
if lightfm is None:
    print("lightfm not installed — skipping evaluation.")
else:
    from lightfm.evaluation import auc_score, precision_at_k, recall_at_k

    test_auc = auc_score(
        lf_model,
        test_interactions=test_interactions,
        train_interactions=train_interactions,
        num_threads=4,
    ).mean()
    test_precision = precision_at_k(
        lf_model,
        test_interactions=test_interactions,
        train_interactions=train_interactions,
        k=10,
        num_threads=4,
    ).mean()
    test_recall = recall_at_k(
        lf_model,
        test_interactions=test_interactions,
        train_interactions=train_interactions,
        k=10,
        num_threads=4,
    ).mean()

    print(f"LightFM WARP — AUC={test_auc:.4f}  Precision@10={test_precision:.4f}  Recall@10={test_recall:.4f}")

lightfm not installed — skipping evaluation.


## Hybrid recommender
Blends LightFM collaborative scores with content cosine similarity.
`alpha` controls CF weight; `(1 - alpha)` controls content weight.

In [7]:
# Prepare mappings to go between external ids and internal indices
if lightfm is None:
    print("lightfm not installed — skipping mapping setup.")
else:
    user_id_map, user_feature_map, item_id_map, item_feature_map = lf_dataset.mapping()
    item_id_inv_map = {inner: raw for raw, inner in item_id_map.items()}
    item_order = [item_id_inv_map[i] for i in range(len(item_id_inv_map))]
    item_index_lookup = {raw_id: idx for idx, raw_id in enumerate(item_order)}

lightfm not installed — skipping mapping setup.


In [8]:
# Reorder similarity matrix to align with LightFM internal ordering
if lightfm is None:
    print("lightfm not installed — skipping similarity reorder.")
else:
    listings_aligned = listings_in_likes.set_index('listing_id').loc[item_order]
    features_scaled_aligned = scaler.transform(listings_aligned[numeric_cols].fillna(0))
    item_similarity = cosine_similarity(features_scaled_aligned)

lightfm not installed — skipping similarity reorder.


In [9]:
def recommend_hybrid(user_id, top_n=10, alpha=0.6):
    """
    Recommend listings by blending collaborative (LightFM) and content similarity scores.
    alpha: weight for CF; (1-alpha) for content.
    Returns list of listing_ids, or raises RuntimeError if lightfm is not installed.
    """
    if lightfm is None:
        raise RuntimeError("lightfm not installed — cannot run hybrid recommender.")
    if user_id not in user_id_map:
        raise ValueError(f'Unknown user_id: {user_id}')
    user_internal = user_id_map[user_id]

    cf_scores = lf_model.predict(user_internal, np.arange(len(item_order)))
    cf_scores = (cf_scores - cf_scores.min()) / (cf_scores.max() - cf_scores.min() + 1e-8)

    liked_items = user_likes_df[user_likes_df['user_id'] == user_id]['listing_id']
    liked_internal = [item_index_lookup[iid] for iid in liked_items if iid in item_index_lookup]
    if liked_internal:
        content_scores = item_similarity[:, liked_internal].mean(axis=1)
    else:
        content_scores = np.zeros(len(item_order))
    content_scores = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min() + 1e-8)

    hybrid_scores = alpha * cf_scores + (1 - alpha) * content_scores

    exclude = set(liked_internal)
    ranked_indices = [idx for idx in np.argsort(-hybrid_scores) if idx not in exclude]
    return [item_id_inv_map[idx] for idx in ranked_indices[:top_n]]

In [10]:
# Example recommendation for the first user
if lightfm is None:
    print("lightfm not installed — skipping hybrid recommendation example.")
else:
    sample_user = users_df['user_id'].iloc[0]
    print('Hybrid recommendations for user', sample_user, ':', recommend_hybrid(sample_user, top_n=5))

lightfm not installed — skipping hybrid recommendation example.
